# 05 - Demonstração (vídeo)
Assistente do Calçada Livre: **RAG + limiar de 'não consta'** com o Qwen2.5-1.5B. Mostra (1) pergunta que consta, (2) pergunta que não consta e (3) explicação de uma ocorrência. Colab: Ambiente de execução > Alterar tipo > GPU T4.

In [ ]:
# Setup
import os, sys, subprocess
REPO = "https://github.com/melo-zz/calcada-livre-genai.git"
if os.path.exists("/content"):
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())
print("pasta de trabalho:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt
%pip uninstall -y -q torchao

In [ ]:
import json, os
from src.kb import carregar_trechos
from src.retrieval import Retriever
from src.llm import HFGen
from src.pipeline import Assistente
from avaliacao.casos import carregar
from src.retrieval import escolher_limiar
os.makedirs(".cache", exist_ok=True)
trechos = carregar_trechos("base/trechos.jsonl")
r = Retriever(trechos, cache=".cache/emb.npy")
limiar = escolher_limiar(carregar(), r)["limiar"]      # mesmo limiar do notebook 02
gen = HFGen()                                           # modelo base (configuração B: RAG simples)
assistente = Assistente(r, lambda m: gen.gerar(m, usar_adapter=False), limiar)
print("pronto | limiar:", round(limiar, 3), "| trechos:", len(trechos))

In [ ]:
def mostrar(pergunta):
    s = assistente.perguntar(pergunta)
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", s["resposta"])
    print("FONTES  :", s["fontes"])
    print("CONSTA  :", s["consta"], "| caminho:", s["caminho"], "| encaminhar para humano:", s["encaminhar_humano"])
    for c in s["contextos"]:
        print(f"   trecho {c['id']}  score={c['score']}")
    print()

## 1. Pergunta que CONSTA na base

In [ ]:
mostrar("Os semáforos para pedestres precisam emitir sinal sonoro?")

## 2. Pergunta que NÃO CONSTA na base (o assistente recusa e encaminha para um humano)

In [ ]:
mostrar("Qual o prazo para a prefeitura consertar um buraco na calçada?")

## 3. Explicação de uma ocorrência (detectada pelo carrinho, pela foto ou pelo app)

In [ ]:
def explicar_ocorrencia(tipo):
    descricoes = {"buraco": "um buraco na calçada", "obstaculo": "um obstáculo no passeio", "piso_quebrado": "piso quebrado ou irregular",
                  "degrau_sem_rampa": "um degrau sem rampa", "trecho_escuro": "um trecho sem iluminação"}
    desc = descricoes.get(tipo, "uma barreira de acessibilidade")
    mostrar(f"Uma pessoa encontrou {desc} em uma calçada. Quais direitos e normas de acessibilidade se aplicam?")
explicar_ocorrencia("degrau_sem_rampa")